# Báo cáo Project
Lớp TTNT-172886, Nhóm G14

## 1. Thông tin chung

### Thành viên
- Võ Hữu Trí Dũng 20235057
- Nguyễn Văn Tiến Dũng 20235306
- Nguyễn Đức Huân 20235338

### Lịch thực hiện
- W2: Đăng ký nhóm 
- W3-4: Đề xuất project (26/9)
- W08: Báo cáo tiến độ giữa kỳ (24/10)
- W15: Hoàn thành và gửi báo cáo project (12/12)
- W16-18: Trình bày project, Q&A

## 2. Đề xuất project (W3)

### Bài toán
Dự án nghiên cứu bài toán **học chính sách chọn nước đi trong cờ vua bằng học có giám sát**. Đầu vào là một trạng thái bàn cờ tiêu chuẩn chứa đầy đủ thông tin để xác định các nước hợp lệ; đầu ra là phân phối xác suất trên các nước đi đó. Khi chơi, hệ thống chọn nước có xác suất cao nhất sau một lần suy luận của mạng, không mở rộng cây tìm kiếm và không sử dụng tự chơi tăng cường.

Dữ liệu học được xây dựng trực tiếp từ các ván đấu chất lượng cao. Mỗi trạng thái ngay trước một lượt đi được ghép với nước thực tế mà người chơi đã thực hiện, tạo thành một cặp trạng thái–nước đi dùng làm mẫu có giám sát. Nước đã chơi phản ánh lựa chọn chiến thuật và chiến lược của người chơi mạnh, nhưng không được xem là nước tối ưu tuyệt đối; vì vậy, độ chính xác dự đoán thể hiện mức độ mô hình học được hành vi trong dữ liệu chứ không trực tiếp khẳng định sức mạnh cờ vua.

Câu hỏi thực nghiệm chính là: **với khoảng 1,5 triệu vị trí được cân bằng theo giai đoạn, mạng ResNet-16×256 có học được một chính sách luôn chọn nước hợp lệ và dự đoán tốt nước đã chơi trên các ván chưa từng xuất hiện trong huấn luyện hay không?** Dự án đồng thời khảo sát ảnh hưởng của biểu diễn canonical và việc cân bằng dữ liệu giữa khai cuộc, trung cuộc, tàn cuộc đối với kết quả ở từng giai đoạn. Phạm vi tập trung vào xây dựng và đánh giá một pipeline học chính sách có giám sát khả thi, không đặt mục tiêu đạt sức mạnh của một engine cờ vua hoàn chỉnh.

### Phương pháp dự kiến
Quy trình thực hiện gồm bốn khối chính: xây dựng dữ liệu, biểu diễn trạng thái và hành động, huấn luyện mạng chính sách, và đánh giá mô hình.

1. **Xây dựng dữ liệu có giám sát**
   - Sử dụng các ván cờ tiêu chuẩn chất lượng cao từ **Lichess Elite Database**, bắt đầu từ dữ liệu tháng 12/2021 và tuân theo tiêu chí tuyển chọn của nguồn.
   - Phân tích PGN bằng thư viện có kiểm tra luật; loại bỏ các ván không thuộc biến thể standard, thiếu chuỗi nước đi, không thể phân tích hoặc không tạo được trạng thái hợp lệ.
   - Từ mỗi ván, trích các cặp $(s_t,a_t)$, trong đó $s_t$ là trạng thái ngay trước lượt thứ $t$ và $a_t$ là nước thực tế đã chơi. Nước đi phải hợp lệ trên đúng trạng thái tương ứng; dữ liệu không cần bước chú giải bổ sung trước khi huấn luyện.
   - Chia dữ liệu theo **ván cờ** với tỷ lệ 90% train, 5% validation và 5% test, sau đó khử trùng lặp vị trí giữa các tập để hạn chế rò rỉ dữ liệu. Dự kiến sử dụng 1.500.000 vị trí, cân bằng thành 500.000 vị trí khai cuộc, 500.000 vị trí trung cuộc và 500.000 vị trí tàn cuộc; mỗi ván đóng góp tối đa một vị trí cho mỗi giai đoạn.

2. **Biểu diễn trạng thái và không gian hành động**
   - Chuẩn hóa bàn cờ theo góc nhìn của bên tới lượt và mã hóa thành tensor $17 \times 8 \times 8$: 12 mặt phẳng quân cờ, 4 mặt phẳng quyền nhập thành và 1 mặt phẳng ô bắt tốt qua đường.
   - Ánh xạ nước đi vào không gian hành động cố định kiểu AlphaZero có kích thước $8 \times 8 \times 73 = 4{,}672$, gồm các nước đi theo hướng, nước mã và các trường hợp phong cấp. Hàm mã hóa và giải mã được kiểm thử cho cả hai màu cùng các nước đặc biệt.
   - Sinh tập nước hợp lệ từ trạng thái gốc và áp dụng **legal move mask** trước softmax. Nhờ đó, xác suất của mọi nước bất hợp lệ bằng 0 và phân phối đầu ra chỉ được chuẩn hóa trên các nước hợp lệ.

3. **Kiến trúc và huấn luyện mô hình**
   - Sử dụng backbone **ResNet-16×256** gồm 16 residual blocks, mỗi block duy trì 256 kênh đặc trưng. Policy head dùng tích chập $1 \times 1$ để chuyển 256 kênh thành 73 kênh, sau đó làm phẳng thành 4.672 logits tương ứng với toàn bộ không gian hành động.
   - Mỗi nước đã chơi là một nhãn one-hot. Mô hình được huấn luyện trực tiếp bằng cross-entropy trên phân phối sau khi áp dụng mặt nạ hợp lệ: $\mathcal{L}_{CE}(s,a^*)=-\log \pi_\theta(a^*\mid s)$.
   - Cấu hình dự kiến gồm AdamW, learning rate $3 \times 10^{-4}$, weight decay $10^{-4}$, cosine decay, batch hiệu dụng 512 và tối đa 8 epoch. Mixed precision và gradient accumulation được sử dụng nếu cần để phù hợp với GPU. Checkpoint được chọn theo negative log-likelihood thấp nhất trên validation set; test set chỉ được sử dụng sau khi cấu hình đã khóa.

4. **Đánh giá và so sánh**
   - Đánh giá trên 75.000 vị trí test thuộc các ván chưa xuất hiện trong train/validation, gồm 25.000 vị trí cho mỗi giai đoạn.
   - Sử dụng các chỉ số **Top-1, Top-3, Top-5 accuracy** và **negative log-likelihood (NLL)** so với nước thực tế trong PGN; báo cáo riêng cho khai cuộc, trung cuộc và tàn cuộc.
   - So sánh chính mô hình huấn luyện trên dữ liệu cân bằng giai đoạn với mô hình đối chứng dùng phân phối giai đoạn tự nhiên, trong khi giữ nguyên số mẫu train, kiến trúc và ngân sách tối ưu. Nếu đủ tài nguyên, khảo sát thêm ảnh hưởng của biểu diễn không canonical hoặc kích thước mạng.
   - Thử chơi ván có kiểm soát giữa các mô hình bằng cùng vị trí khởi đầu và giới hạn suy luận, đổi màu theo cặp, thống kê thắng/hòa/thua và lưu PGN để phân tích lỗi.

### Phân công
- **Võ Hữu Trí Dũng — Phụ trách dữ liệu:**
  - Thu thập và kiểm tra các ván cờ từ Lichess Elite Database; xây dựng bộ lọc để loại bỏ dữ liệu không hợp lệ hoặc không đáp ứng tiêu chí của dự án.
  - Trích các cặp trạng thái–nước đi từ PGN, xác minh tính hợp lệ của nước đi và lưu các thông tin cần thiết như mã ván, FEN, ply, nước đã chơi và giai đoạn.
  - Phân chia dữ liệu theo ván thành train/validation/test, khử trùng lặp giữa các tập và lấy mẫu cân bằng theo khai cuộc, trung cuộc, tàn cuộc.
  - Xây dựng quy trình chuẩn hóa trạng thái, mã hóa nước đi và kiểm thử các trường hợp đặc biệt như nhập thành, bắt tốt qua đường và phong cấp; quản lý shard, manifest và hash dữ liệu.

- **Nguyễn Văn Tiến Dũng — Phụ trách huấn luyện:**
  - Cài đặt mạng chính sách ResNet-16×256, policy head, legal move masking và hàm mất mát cross-entropy trên nước đã chơi.
  - Xây dựng các cấu hình thí nghiệm chính, gồm mô hình huấn luyện trên dữ liệu cân bằng giai đoạn và mô hình đối chứng dùng phân phối giai đoạn tự nhiên.
  - Thiết lập optimizer, learning-rate scheduler, mixed precision và gradient accumulation; benchmark để lựa chọn microbatch phù hợp với tài nguyên GPU.
  - Theo dõi loss, NLL và accuracy trên validation set; lưu checkpoint, cấu hình và log để quá trình huấn luyện có thể tiếp tục và tái lập.

- **Nguyễn Đức Huân — Phụ trách đánh giá:**
  - Xây dựng pipeline suy luận và đánh giá thống nhất trên test set, sử dụng cùng cách mã hóa và mặt nạ nước hợp lệ cho tất cả mô hình.
  - Tính toán Top-1, Top-3, Top-5 accuracy và negative log-likelihood; phân tích riêng kết quả ở khai cuộc, trung cuộc và tàn cuộc.
  - Tổ chức các ván đấu có kiểm soát nếu đủ tài nguyên, sử dụng cùng vị trí khởi đầu và giới hạn suy luận, đổi màu theo cặp và thống kê thắng/hòa/thua.
  - So sánh các cấu hình, tổng hợp bảng kết quả, trực quan hóa số liệu và phân tích các trường hợp nhãn có thứ hạng thấp hoặc phát sinh lỗi trong pipeline.

## 3. Tiến độ giữa kỳ (W8)

### Chương trình


### Phân tích kết quả, vấn đề gặp phải


## 4. Cập nhật kết quả cuối kỳ (W15)

### Chi tiết phương pháp, dữ liệu 


### Chương trình


### Phân tích, đánh giá kết quả


### Cập nhật phân công, khối lượng công việc
<!-- công việc của các thành viên, tỷ lệ đóng góp của các thành viên -->
